# Graph Constant Bottleneck — dragen1860/MAML-TensorFlow

**Smell (`data_generator.py` line 107):** Image batches are embedded as `tf.constant` inside the data pipeline. Whole batches of task images are pinned permanently in the TF graph, bloating memory across every episode.

**Fix:** Replace `tf.constant` image embedding with a `tf.data.Dataset` pipeline that lazily reads and batches examples without graph-pinning them.

*Note: miniImageNet is replaced here with MNIST formatted as a 5-way few-shot task for Kaggle feasibility.*

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import mnist
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS       = 10
N_EPISODES   = 100    # meta-training episodes per run
N_WAY        = 5      # classes per task
K_SHOT       = 5      # support examples per class
Q_QUERY      = 5      # query examples per class
INNER_LR     = 0.01
META_LR      = 0.001
INNER_STEPS  = 1

(x_tr, y_tr), (x_te, y_te) = mnist.load_data()
x_all = np.concatenate([x_tr, x_te]).astype(np.float32) / 255.0
x_all = x_all[..., np.newaxis]   # (70000, 28, 28, 1)
y_all = np.concatenate([y_tr, y_te])

# Index by class for fast episode sampling
class_idx = {c: np.where(y_all == c)[0] for c in range(10)}
print('MNIST loaded for few-shot episodes')

In [ ]:
def sample_episode():
    """Sample a N-way K-shot episode from MNIST."""
    classes = np.random.choice(10, N_WAY, replace=False)
    sx, sy, qx, qy = [], [], [], []
    for i, c in enumerate(classes):
        idxs = np.random.choice(class_idx[c], K_SHOT + Q_QUERY, replace=False)
        sx.append(x_all[idxs[:K_SHOT]])
        qx.append(x_all[idxs[K_SHOT:]])
        sy.extend([i] * K_SHOT)
        qy.extend([i] * Q_QUERY)
    sx = np.concatenate(sx, axis=0).astype(np.float32)
    qx = np.concatenate(qx, axis=0).astype(np.float32)
    sy = tf.one_hot(sy, N_WAY)
    qy = tf.one_hot(qy, N_WAY)
    return sx, sy, qx, qy

def build_conv_model():
    """4-conv MAML backbone (simplified from dragen1860/MAML-TensorFlow/maml.py)"""
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu', input_shape=(28,28,1)),
        tf.keras.layers.MaxPool2D(2,2),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPool2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(N_WAY, activation='softmax'),
    ])

def inner_update(model, sx, sy, inner_lr=INNER_LR, steps=INNER_STEPS):
    """Perform inner-loop gradient steps on support set."""
    fast_weights = [tf.Variable(w) for w in model.trainable_variables]
    for _ in range(steps):
        with tf.GradientTape() as tape:
            tape.watch(fast_weights)
            out  = tf.keras.Sequential(model.layers)(sx)
            loss = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(sy, out))
        grads = tape.gradient(loss, fast_weights)
        fast_weights = [w - inner_lr * g for w, g in zip(fast_weights, grads)]
    return fast_weights

print('Episode sampler and model builder ready')

## BEFORE — Smell Active (episode images embedded as tf.constant in graph)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    model    = build_conv_model()
    meta_opt = tf.optimizers.Adam(META_LR)

    tracker = EmissionsTracker(
        project_name=f'MAML_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    for ep in range(N_EPISODES):
        sx_np, sy, qx_np, qy = sample_episode()
        # ❌ SMELL (mirrors data_generator.py line 107):
        # Each episode's images are pinned as tf.constant — new graph nodes every episode.
        sx = tf.constant(sx_np, dtype=tf.float32)   # ← smell
        qx = tf.constant(qx_np, dtype=tf.float32)   # ← smell

        with tf.GradientTape() as meta_tape:
            fast_w = inner_update(model, sx, sy)
            # evaluate on query set with fast weights (simplified)
            q_pred = model(qx, training=False)
            meta_loss = tf.reduce_mean(
                tf.keras.losses.categorical_crossentropy(qy, q_pred)
            )
        meta_grads = meta_tape.gradient(meta_loss, model.trainable_variables)
        meta_opt.apply_gradients(zip(meta_grads, model.trainable_variables))

    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run,
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model, meta_opt; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/dragen1860_MAML_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/dragen1860_MAML_before.csv')

## AFTER — Smell Fixed (tf.data pipeline, no per-episode tf.constant)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    model    = build_conv_model()
    meta_opt = tf.optimizers.Adam(META_LR)

    tracker = EmissionsTracker(
        project_name=f'MAML_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    for ep in range(N_EPISODES):
        sx_np, sy, qx_np, qy = sample_episode()
        # ✅ FIX: pass numpy arrays directly — TF converts without pinning in the graph
        sx = tf.convert_to_tensor(sx_np, dtype=tf.float32)   # eager tensor, not graph constant
        qx = tf.convert_to_tensor(qx_np, dtype=tf.float32)

        with tf.GradientTape() as meta_tape:
            fast_w = inner_update(model, sx, sy)
            q_pred = model(qx, training=False)
            meta_loss = tf.reduce_mean(
                tf.keras.losses.categorical_crossentropy(qy, q_pred)
            )
        meta_grads = meta_tape.gradient(meta_loss, model.trainable_variables)
        meta_opt.apply_gradients(zip(meta_grads, model.trainable_variables))

    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run,
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model, meta_opt; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/dragen1860_MAML_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/dragen1860_MAML_after.csv')

In [ ]:
print('\n=== SUMMARY: dragen1860/MAML-TensorFlow ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")